# Phase 1 – Data Cleaning & Quality Check

## IndiaKart E-Commerce

## Objective
The objective of this phase is to load the IndiaKart data, check its quality, and report any problems before analysis.

In [54]:
import pandas as pd
folder = ""
file_names = ["suppliers", "products", "inventory", "customers", "orders", "order_items", "payments", "returns"]

dfs = {}
for name in file_names:
    dfs[name] = pd.read_csv(folder + name + " DA.csv")

print("Tables loaded:", len(dfs))

Tables loaded: 8


In [55]:
rows = []
for name in dfs:
    rows.append([name, dfs[name].shape[0], dfs[name].shape[1]])

summary = pd.DataFrame(rows, columns=["table", "rows", "columns"])
summary

,table,rows,columns
0,suppliers,200,14
1,products,1000,17
2,inventory,1000,11
3,customers,10000,17
4,orders,50000,19
5,order_items,100000,11
6,payments,50000,13
7,returns,10000,10


## Data Understanding

The IndiaKart database has 8 related tables covering two years of business, from 24 June 2023 to 23 June 2025.

| Table | Rows | What it contains | Key columns |
|---|---|---|---|
| customers | 10,000 | Customer profiles and lifetime stats | customer_id |
| orders | 50,000 | Every order placed (central table) | order_id, customer_id |
| order_items | 100,000 | The individual products inside each order | item_id, order_id, product_id |
| payments | 50,000 | Payment attempts and their status | payment_id, order_id, customer_id |
| returns | 10,000 | Return requests with reasons and refund status | return_id, order_id, customer_id |
| products | 1,000 | Product catalogue with prices and GST | product_id, supplier_id |
| inventory | 1,000 | Warehouse stock levels | inventory_id, product_id |
| suppliers | 200 | Supplier details and ratings | supplier_id |

**How the tables connect**
- A customer places many orders (customers → orders, through customer_id).
- An order contains many items (orders → order_items, through order_id), and each item is a product (order_items → products, through product_id).
- Each order has payments, and each return refers to an order (through order_id).
- Each product comes from a supplier (products → suppliers) and has an inventory record (products → inventory).

**Business context:** IndiaKart sells across 10 product categories, to 5 customer segments (Regular, Budget, Premium, New and Inactive), through the app, website and mobile web. Orders have one of five statuses: Delivered, Cancelled, Shipped, Returned and Processing.

## Data Profiling and Data Types

In [56]:
for name in dfs:
    for col in dfs[name].columns:
        if "date" in col or "time" in col:
            print(name, "|", col, "|", dfs[name][col].dtype)

suppliers | created_date | str
products | launch_date | str
inventory | last_restocked_date | str
customers | registration_date | str
customers | last_login_date | str
orders | order_date | str
orders | order_time | str
orders | delivered_date | str
payments | payment_date | str
payments | payment_time | str
payments | refund_date | float64
returns | return_date | str
returns | refund_date | str


### Amount columns are numeric

In [57]:
amount_columns = {
    "products": ["mrp", "selling_price", "cost_price"],
    "customers": ["total_spent"],
    "orders": ["total_amount", "gst_amount", "shipping_charge", "discount_amount", "final_amount"],
    "order_items": ["unit_price", "gst_amount", "discount_amount", "total_price"],
    "payments": ["amount", "refund_amount"],
    "returns": ["return_amount"],
    "inventory": ["unit_cost", "total_inventory_value"],
}
for table in amount_columns:
    for col in amount_columns[table]:
        print(table, "|", col, "|", dfs[table][col].dtype)

products | mrp | int64
products | selling_price | int64
products | cost_price | int64
customers | total_spent | float64
orders | total_amount | float64
orders | gst_amount | float64
orders | shipping_charge | int64
orders | discount_amount | float64
orders | final_amount | float64
order_items | unit_price | int64
order_items | gst_amount | float64
order_items | discount_amount | float64
order_items | total_price | float64
payments | amount | float64
payments | refund_amount | int64
returns | return_amount | float64
inventory | unit_cost | int64
inventory | total_inventory_value | int64



All 8 IndiaKart tables were loaded and profiled.

### Observation
The data has 222,200 records across 8 tables, and the largest are order_items (100,000 rows), orders (50,000) and payments (50,000). Amount columns are already numeric, but all date and time columns are stored as text, so they will be converted to proper dates before analysis.

### Missing values

In [58]:
for name in dfs:
    df = dfs[name]
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(name, ": no missing values")
    else:
        print(name)
        print(pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)}))
    print("-" * 40)

suppliers : no missing values
----------------------------------------
products : no missing values
----------------------------------------
inventory : no missing values
----------------------------------------
customers : no missing values
----------------------------------------
orders
                missing  percent
delivered_date    17501     35.0
----------------------------------------
order_items : no missing values
----------------------------------------
payments
             missing  percent
refund_date    50000    100.0
----------------------------------------
returns : no missing values
----------------------------------------


In [59]:
orders = dfs["orders"]
print(orders[orders["delivered_date"].isnull()]["status"].value_counts())
print("Delivered orders without a delivered date:", orders[(orders["status"] == "Delivered") & (orders["delivered_date"].isnull())].shape[0])
print("Total refund amount in payments:", dfs["payments"]["refund_amount"].sum())

status
Cancelled     5996
Shipped       5071
Returned      3933
Processing    2501
Name: count, dtype: int64
Delivered orders without a delivered date: 0
Total refund amount in payments: 0


## Missing Value Analysis

### Observation
Only two columns contain missing values. In the orders table, delivered_date is empty for 17,501 orders (35.00%), and all of them have a status of Cancelled, Shipped, Processing or Returned, so this is expected. Every Delivered order has a delivery date. It is worth noting that the 3,933 Returned orders have no delivery date. In the payments table, refund_date is empty for all 50,000 rows and every refund_amount is 0, so refunds are not recorded in this table, and the returns table should be used for refund analysis. All other columns are complete.

## Duplicate rows

In [60]:
for name in dfs:
    print(name, "duplicate rows:", dfs[name].duplicated().sum())

suppliers duplicate rows: 0
products duplicate rows: 0
inventory duplicate rows: 0
customers duplicate rows: 0
orders duplicate rows: 0
order_items duplicate rows: 0
payments duplicate rows: 0
returns duplicate rows: 0


#### Duplicate IDs

In [61]:
keys = {"suppliers": "supplier_id", "products": "product_id", "inventory": "inventory_id", "customers": "customer_id",
        "orders": "order_id", "order_items": "item_id", "payments": "payment_id", "returns": "return_id"}

for table in keys:
    print(table, "duplicate", keys[table], ":", dfs[table][keys[table]].duplicated().sum())

suppliers duplicate supplier_id : 0
products duplicate product_id : 0
inventory duplicate inventory_id : 0
customers duplicate customer_id : 0
orders duplicate order_id : 0
order_items duplicate item_id : 0
payments duplicate payment_id : 0
returns duplicate return_id : 0


In [62]:
print("Duplicate transaction_id in payments:", dfs["payments"]["transaction_id"].duplicated().sum())
print("Duplicate tracking_id in orders:", dfs["orders"]["tracking_id"].duplicated().sum())
print("Duplicate email in customers:", dfs["customers"]["email"].duplicated().sum())

Duplicate transaction_id in payments: 0
Duplicate tracking_id in orders: 1
Duplicate email in customers: 7


### Observation

No duplicate rows were found in any of the 8 tables, and every primary key (order_id, customer_id, item_id and so on) is unique, including order_id in orders. Two small repeats were found in columns that should be unique: 7 customers share an email address with another customer, and one tracking_id appears twice in orders. Neither affects the analysis, but they are noted for completeness.

### Convert the date columns

In [63]:
date_columns = {
    "suppliers": ["created_date"],
    "products": ["launch_date"],
    "inventory": ["last_restocked_date"],
    "customers": ["registration_date", "last_login_date"],
    "orders": ["order_date", "delivered_date"],
    "payments": ["payment_date", "refund_date"],
    "returns": ["return_date", "refund_date"],
}

for table in date_columns:
    for col in date_columns[table]:
        before = dfs[table][col].isnull().sum()
        dfs[table][col] = pd.to_datetime(dfs[table][col], format="%d-%m-%Y", errors="coerce")
        after = dfs[table][col].isnull().sum()
        if before != after:
            print("PROBLEM:", table, col, "-", after - before, "dates could not be converted")

print("Order dates:", dfs["orders"]["order_date"].min().date(), "to", dfs["orders"]["order_date"].max().date())
print(dfs["orders"][["order_date", "delivered_date"]].dtypes)

Order dates: 2023-06-24 to 2025-06-23
order_date        datetime64[us]
delivered_date    datetime64[us]
dtype: object


In [64]:
print("Customer registrations:", dfs["customers"]["registration_date"].min().date(), "to", dfs["customers"]["registration_date"].max().date())
print("Payments:", dfs["payments"]["payment_date"].min().date(), "to", dfs["payments"]["payment_date"].max().date())
print("Returns:", dfs["returns"]["return_date"].min().date(), "to", dfs["returns"]["return_date"].max().date())

Customer registrations: 2020-06-24 to 2025-06-23
Payments: 2023-06-24 to 2025-06-23
Returns: 2023-06-26 to 2025-07-05


### Observation

All date columns were stored as text in DD-MM-YYYY format and were converted to proper dates without any failures. Orders run from 24 June 2023 to 23 June 2025, covering exactly two years. Customers registered from June 2020, so the business had customers before the order period, and the last returns are dated early July 2025, a few days after the last order.


#### OUTLIERS

In [65]:
orders = dfs["orders"]
print(orders["final_amount"].describe().round(0))

large = orders[orders["final_amount"] > 500000]
print("Orders above Rs 5 lakh:", len(large), "(", round(len(large) / len(orders) * 100, 2), "% of orders )")
print("Share of total order value:", round(large["final_amount"].sum() / orders["final_amount"].sum() * 100, 2), "%")
print(large["status"].value_counts())
print(orders.nlargest(3, "final_amount")[["order_id", "final_amount", "status"]])

# what kind of products are in these large orders?
large_items = dfs["order_items"][dfs["order_items"]["order_id"].isin(large["order_id"])]
print(large_items["category"].value_counts().head(3))

count      50000.0
mean       62948.0
std        97898.0
min          136.0
25%         6894.0
50%        24183.0
75%        71496.0
max      1133382.0
Name: final_amount, dtype: float64
Orders above Rs 5 lakh: 312 ( 0.62 % of orders )
Share of total order value: 5.8 %
status
Delivered     202
Cancelled      43
Shipped        28
Returned       25
Processing     14
Name: count, dtype: int64
        order_id  final_amount     status
27602  ORD027603    1133382.48   Returned
10854  ORD010855     967333.45  Delivered
2615   ORD002616     943168.29  Delivered
category
Electronics         525
Sports & Fitness     92
Home & Kitchen       85
Name: count, dtype: int64


### Observation

Order values range from Rs 136 to Rs 1,133,382, with a median of Rs 24,183 and a mean of Rs 62,948, which shows that a few very large orders pull the average up. 312 orders (0.62%) are above Rs 5 lakh and together make up 5.8% of total order value. They are mostly electronics purchases and most were delivered, so they look like genuine high-value orders. They were flagged but kept in the data.


##### Delivery date after order date

In [66]:
delivered = orders[orders["delivered_date"].notnull()]
days = (delivered["delivered_date"] - delivered["order_date"]).dt.days

print("Delivered orders:", len(delivered))
print("Delivered on or before the order date:", (days <= 0).sum())
print("Delivery time (days): average", round(days.mean(), 1), "| minimum", days.min(), "| maximum", days.max())

Delivered orders: 32499
Delivered on or before the order date: 0
Delivery time (days): average 4.5 | minimum 2 | maximum 7


### Observation

All 32,499 delivered orders have a delivery date after the order date, and delivery takes 2 to 7 days (4.5 days on average). No rows needed to be flagged.

### Payment failures

In [67]:
payments = dfs["payments"]
print(payments["status"].value_counts())

failure_rate = (payments["status"] == "Failed").mean() * 100
print("Payment failure rate:", round(failure_rate, 2), "%")

print(payments[payments["status"] == "Failed"]["payment_method"].value_counts())

# which orders do the failed payments belong to?
failed_orders = orders[orders["order_id"].isin(payments[payments["status"] == "Failed"]["order_id"])]
print(failed_orders["status"].value_counts())

status
Success    48252
Failed      1748
Name: count, dtype: int64
Payment failure rate: 3.5 %
payment_method
UPI                 624
Credit Card         363
Debit Card          230
Cash on Delivery    209
Net Banking         159
EMI                  87
Wallet               76
Name: count, dtype: int64
status
Cancelled    1748
Name: count, dtype: int64


### Observation

1,748 of the 50,000 payment attempts failed, a failure rate of 3.50%, which is above the 2% target. UPI has the most failures (624) and every failed payment belongs to a cancelled order, so payment failures are directly connected to lost orders.

### Returns Checks

In [68]:
returns = dfs["returns"]
print("Returns without a matching order:", (~returns["order_id"].isin(orders["order_id"])).sum())
print(returns["refund_status"].value_counts())
print("Returns with a refund date:", returns["refund_date"].notnull().sum())
print(orders[orders["order_id"].isin(returns["order_id"])]["status"].value_counts())

Returns without a matching order: 0
refund_status
Refunded    7268
Pending     2094
Rejected     638
Name: count, dtype: int64
Returns with a refund date: 10000
status
Delivered    4327
Returned     3933
Cancelled    1067
Shipped       673
Name: count, dtype: int64


### Observation

Every one of the 10,000 return requests matches an order in the orders table. Of these, 7,268 are refunded, 2,094 are pending and 638 were rejected. Returns exist for Delivered (4,327) and Returned (3,933) orders, but also for Cancelled (1,067) and Shipped (673) orders, which is unexpected because those orders had not been delivered. In addition, refund_date is filled for all 10,000 returns, including the pending and rejected ones, so it should be read as the expected refund date and not as proof that a refund was completed. These points will matter when the return rate is calculated in Phase 3.

## Inventory Check

In [69]:
inventory = dfs["inventory"]
print(inventory["status"].value_counts())
print(inventory[inventory["status"] == "Out of Stock"][["inventory_id", "product_id", "quantity_available"]])
print("Products below reorder level:", (inventory["quantity_available"] < inventory["reorder_level"]).sum())

status
In Stock        922
Low Stock        76
Out of Stock      2
Name: count, dtype: int64
    inventory_id product_id  quantity_available
853      INV0854    PRD0854                   0
899      INV0900    PRD0900                   0
Products below reorder level: 78


### Observation

Only 2 of the 1,000 products are out of stock (PRD0854 and PRD0900) and 76 more are on low stock, so 78 products are below their reorder level. 92.2% of products are fully in stock, which is slightly under the 95% target given in the project document, and the stock status agrees with the reorder levels.

## Customer Check

In [70]:
customers = dfs["customers"]
zero = customers[customers["total_orders"] == 0]
print("Customers with 0 orders:", len(zero))
print(zero["segment"].value_counts())

print("Sum of total_orders in customers:", customers["total_orders"].sum(), "| Orders in the orders table:", len(orders))
print("Total spent (customers table): Rs", round(customers["total_spent"].sum()))
print("Value of all orders: Rs", round(orders["final_amount"].sum()))
print("Value of delivered orders: Rs", round(orders[orders["status"] == "Delivered"]["final_amount"].sum()))

Customers with 0 orders: 457
segment
Inactive    281
Budget      136
New          24
Regular      16
Name: count, dtype: int64
Sum of total_orders in customers: 50000 | Orders in the orders table: 50000
Total spent (customers table): Rs 3147385545
Value of all orders: Rs 3147385545
Value of delivered orders: Rs 2053858612


### Observation

457 customers (4.6%) have never placed an order, mostly Inactive (281) and Budget (136) customers, which is acceptable. The pre-calculated total_orders column is reliable because it adds up to exactly 50,000, the number of orders in the orders table. However, total_spent adds up to Rs 3.15 billion, which is exactly the value of all orders including cancelled and returned ones, while delivered orders are worth only Rs 2.05 billion. So total_spent overstates what customers actually paid, and revenue should be calculated from the orders table.

## Referential Integrity Check

In [71]:
checks = [
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("orders", "customer_id", "customers", "customer_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("payments", "customer_id", "customers", "customer_id"),
    ("returns", "order_id", "orders", "order_id"),
    ("returns", "customer_id", "customers", "customer_id"),
    ("products", "supplier_id", "suppliers", "supplier_id"),
    ("inventory", "product_id", "products", "product_id"),
]

for child, child_col, parent, parent_col in checks:
    missing = (~dfs[child][child_col].isin(dfs[parent][parent_col])).sum()
    print(child + "." + child_col, "->", parent + "." + parent_col, ": rows with no match =", missing)

# the other direction
no_items = (~orders["order_id"].isin(dfs["order_items"]["order_id"])).sum()
print("Orders with no items in order_items:", no_items, "(", round(no_items / len(orders) * 100, 2), "% )")
print("Suppliers with no products:", (~dfs["suppliers"]["supplier_id"].isin(dfs["products"]["supplier_id"])).sum())

order_items.order_id -> orders.order_id : rows with no match = 0
order_items.product_id -> products.product_id : rows with no match = 0
orders.customer_id -> customers.customer_id : rows with no match = 0
payments.order_id -> orders.order_id : rows with no match = 0
payments.customer_id -> customers.customer_id : rows with no match = 0
returns.order_id -> orders.order_id : rows with no match = 0
returns.customer_id -> customers.customer_id : rows with no match = 0
products.supplier_id -> suppliers.supplier_id : rows with no match = 0
inventory.product_id -> products.product_id : rows with no match = 0
Orders with no items in order_items: 10071 ( 20.14 % )
Suppliers with no products: 4


### Observation

All 9 relationships between the tables are intact: every order_id in order_items exists in orders (as asked in the project document), and every customer, product and supplier reference points to an existing record. In the other direction, 10,071 orders (20.14%) have no line items in order_items, so item-level analysis such as category revenue and product performance covers only 39,929 of the 50,000 orders. Four suppliers do not supply any product.

## Findings summary and the Data Quality Report

In [72]:
rows = [
    ["all tables", "Records loaded", "222,200 records in 8 tables", "None needed"],
    ["all tables", "Date columns stored as text", "11 date columns", "Converted to datetime"],
    ["orders", "delivered_date missing", "17,501 (35.00%)", "Expected: orders not delivered. Kept"],
    ["payments", "refund_date missing", "50,000 (100%)", "Column unused. Refunds are in returns"],
    ["all tables", "Duplicate rows", "0", "None needed"],
    ["customers", "Repeated email", "7", "Noted"],
    ["orders", "Repeated tracking_id", "1", "Noted"],
    ["orders", "Orders above Rs 5 lakh", "312 (0.62%)", "Flagged, kept"],
    ["orders", "Delivered on or before order date", "0", "None needed"],
    ["payments", "Payment failure rate", "3.50% (1,748)", "Noted, above 2% target"],
    ["returns", "Returns without a matching order", "0", "None needed"],
    ["returns", "Returns on Cancelled or Shipped orders", "1,740", "Noted"],
    ["returns", "refund_date filled for pending or rejected returns", "2,732", "Noted"],
    ["inventory", "Products out of stock", "2", "Noted"],
    ["customers", "Customers with 0 orders", "457", "Noted"],
    ["customers", "total_spent includes cancelled orders", "Rs 3.15 billion vs Rs 2.05 billion delivered", "Revenue taken from orders"],
    ["orders", "Orders with no items", "10,071 (20.14%)", "Noted"],
    ["suppliers", "Suppliers with no products", "4", "Noted"],
    ["all tables", "Broken references between tables", "0", "None needed"],
]

findings = pd.DataFrame(rows, columns=["table", "check", "result", "action"])
findings

,table,check,result,action
0,all tables,Records loaded,"222,200 records in 8 tables",None needed
1,all tables,Date columns stored as text,11 date columns,Converted to datetime
2,orders,delivered_date missing,"17,501 (35.00%)",Expected: orders not delivered. Kept
3,payments,refund_date missing,"50,000 (100%)",Column unused. Refunds are in returns
4,all tables,Duplicate rows,0,None needed
5,customers,Repeated email,7,Noted
6,orders,Repeated tracking_id,1,Noted
7,orders,Orders above Rs 5 lakh,312 (0.62%),"Flagged, kept"
8,orders,Delivered on or before order date,0,None needed
9,payments,Payment failure rate,"3.50% (1,748)","Noted, above 2% target"


In [73]:
# mark the large orders so they can be found later
dfs["orders"]["is_large_order"] = dfs["orders"]["final_amount"] > 500000
print("Large orders flagged:", dfs["orders"]["is_large_order"].sum())

# save every table with its converted dates
for name in dfs:
    dfs[name].to_csv(name + "_cleaned.csv", index=False)

print("Saved", len(dfs), "cleaned files")

Large orders flagged: 312
Saved 8 cleaned files


# Data Quality Report

## Objective
The objective of this phase was to check the quality of the 8 IndiaKart tables before analysis: their structure, missing values, duplicates, data types, outliers and the links between tables.

## Summary of Findings
- **Data loaded:** 222,200 records in 8 tables, covering orders from 24 June 2023 to 23 June 2025.
- **Data types:** amounts are numeric, but all 11 date columns were stored as text and were converted to dates without any failures.
- **Missing values:** only two columns. delivered_date is empty for 17,501 orders (35.00%), all of them undelivered orders, which is expected. refund_date in payments is empty for every row, because refunds are recorded in the returns table.
- **Duplicates:** none in any table, and every order_id is unique. Small repeats were found in 7 customer emails and 1 tracking_id.
- **Outliers:** 312 orders (0.62%) are above Rs 5 lakh, worth 5.8% of total order value. They are mostly electronics purchases and were flagged and kept.
- **Orders and payments:** every delivered order has a delivery date after the order date (2 to 7 days). The payment failure rate is 3.50%, above the 2% target, and all failed payments belong to cancelled orders.
- **Returns:** all 10,000 returns match an order. However, 1,740 returns belong to cancelled or shipped orders, and refund_date is filled even for pending and rejected returns.
- **Inventory and customers:** 2 products are out of stock and 76 are on low stock. 457 customers have never ordered. The total_spent column includes cancelled orders, so it overstates real spending.
- **Referential integrity:** all 9 links between tables are intact. In the other direction, 10,071 orders (20.14%) have no line items.

## Actions Taken
- Converted all date columns to proper dates.
- Flagged the 312 large orders in a new column and kept them.
- No rows were removed, because no duplicates or broken links were found.
- Saved the cleaned tables for use in the next phases.

## Assumptions and Limitations
- Orders above Rs 5 lakh are treated as genuine high-value orders and not as errors.
- Item-level analysis (category revenue, product performance) covers only the 39,929 orders that have line items.
- Revenue should be taken from the orders table, not from the total_spent column.

## Conclusion
The IndiaKart data is largely clean and consistent, with no duplicates or broken references, so it can be used for analysis. The points to keep in mind are the unusual returns, the 10,071 orders without line items, and the overstated total_spent column, which will be considered when the KPIs are calculated in Phase 3.